In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, accuracy_score, f1_score, confusion_matrix, precision_score, recall_score, classification_report,roc_auc_score,roc_curve,RocCurveDisplay
import os
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from tqdm import tqdm
os.chdir('/home/pgcp-ai/MachineLearning/Cases/Glass_Identification/')

In [2]:
glass = pd.read_csv("Glass.csv")
glass

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.00,0.0,building_windows_float_processed
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.00,0.0,building_windows_float_processed
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.00,0.0,building_windows_float_processed
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.00,0.0,building_windows_float_processed
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.00,0.0,building_windows_float_processed
...,...,...,...,...,...,...,...,...,...,...
209,1.51623,14.14,0.00,2.88,72.61,0.08,9.18,1.06,0.0,headlamps
210,1.51685,14.92,0.00,1.99,73.06,0.00,8.40,1.59,0.0,headlamps
211,1.52065,14.36,0.00,2.02,73.42,0.00,8.44,1.64,0.0,headlamps
212,1.51651,14.38,0.00,1.94,73.61,0.00,8.48,1.57,0.0,headlamps


In [3]:
glass.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 214 entries, 0 to 213
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   RI      214 non-null    float64
 1   Na      214 non-null    float64
 2   Mg      214 non-null    float64
 3   Al      214 non-null    float64
 4   Si      214 non-null    float64
 5   K       214 non-null    float64
 6   Ca      214 non-null    float64
 7   Ba      214 non-null    float64
 8   Fe      214 non-null    float64
 9   Type    214 non-null    object 
dtypes: float64(9), object(1)
memory usage: 16.8+ KB


In [4]:
glass.isna().sum()

RI      0
Na      0
Mg      0
Al      0
Si      0
K       0
Ca      0
Ba      0
Fe      0
Type    0
dtype: int64

In [5]:
le = LabelEncoder()
glass["Type"] = le.fit_transform(glass["Type"])

In [6]:
X, y = glass.drop("Type", axis = 1), glass["Type"]

In [7]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,stratify=y,random_state=26)

In [8]:
ss = StandardScaler()
X_train = ss.fit_transform(X_train)
X_test = ss.transform(X_test)

In [9]:
Cs = np.linspace(0.001,5,20)
dfs = ['ovo', 'ovr']
scores = []
for c in tqdm(Cs):
    for f in dfs:
        svm = SVC(kernel='linear',C=c,probability=True,random_state=26, decision_function_shape=f)
        svm.fit(X_train,y_train)
        y_pred = svm.predict(X_test)
        acc = accuracy_score(y_test,y_pred)
        y_pred_prob = svm.predict_proba(X_test)
        log_loss_score = log_loss(y_test,y_pred_prob)
        scores.append([c,f, acc,log_loss_score])

100%|███████████████████████████████████████████| 20/20 [00:00<00:00, 40.82it/s]


In [10]:
df_scores = pd.DataFrame(scores,columns=['C','Decision Function', 'Accuracy Score','Log Loss'])
df_scores.sort_values('Log Loss',ascending=True)

,C,Decision Function,Accuracy Score,Log Loss
6,0.790316,ovo,0.569231,0.951070
7,0.790316,ovr,0.569231,0.951070
10,1.316526,ovo,0.584615,0.953640
11,1.316526,ovr,0.584615,0.953640
8,1.053421,ovo,0.569231,0.953687
9,1.053421,ovr,0.569231,0.953687
13,1.579632,ovr,0.630769,0.954680
12,1.579632,ovo,0.630769,0.954680
17,2.105842,ovr,0.600000,0.955157
16,2.105842,ovo,0.600000,0.955157


In [11]:
Cs = np.linspace(0.001,5,20)
Gs = np.linspace(0.001, 5, 20)
dfs = ['ovo', 'ovr']
scores = []
for c in tqdm(Cs):
    for f in dfs: 
        for g in Gs:
            svm = SVC(kernel='rbf',C=c,probability=True,gamma = g,random_state=26,decision_function_shape=f)
            svm.fit(X_train,y_train)
            y_pred = svm.predict(X_test)
            acc = accuracy_score(y_test,y_pred)
            y_pred_prob = svm.predict_proba(X_test)
            log_loss_score = log_loss(y_test,y_pred_prob)
            scores.append([c,f,g,acc,log_loss_score])

100%|███████████████████████████████████████████| 20/20 [00:12<00:00,  1.55it/s]


In [12]:
df_scores = pd.DataFrame(scores, columns = ['C','Decision Function','G', 'Accuracy Score', 'Log Loss'])

In [13]:
df_scores.sort_values(['Log Loss', 'Accuracy Score'], ascending = [True, False])

,C,Decision Function,G,Accuracy Score,Log Loss
201,1.316526,ovo,0.264105,0.738462,0.769268
221,1.316526,ovr,0.264105,0.738462,0.769268
161,1.053421,ovo,0.264105,0.723077,0.771740
181,1.053421,ovr,0.264105,0.723077,0.771740
241,1.579632,ovo,0.264105,0.738462,0.773397
...,...,...,...,...,...
37,0.001000,ovr,4.473789,0.353846,1.350395
18,0.001000,ovo,4.736895,0.353846,1.359203
38,0.001000,ovr,4.736895,0.353846,1.359203
19,0.001000,ovo,5.000000,0.353846,1.366651
